# AI-Powered Employee Attrition Intelligence System
## Interactive Exploration Notebook

This notebook walks through the full pipeline:
1. Data Loading & HR Feature Engineering
2. Exploratory Data Analysis
3. XGBoost Training with Optuna
4. SHAP Explainability
5. Risk Scoring & Cost Model
6. What-If Scenario Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, fbeta_score, roc_auc_score, confusion_matrix

sns.set_theme(style='whitegrid')
shap.initjs()

## 1. Load Data & Engineer HR Features

In [ ]:
df = pd.read_csv('../datasets/HR-Employee-Attrition.csv')
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

# -- HR Domain Features --
df['Avg_Role_Income'] = df.groupby('JobRole')['MonthlyIncome'].transform('mean')
df['Compa_Ratio'] = df['MonthlyIncome'] / df['Avg_Role_Income']
df['Promotion_Stagnation'] = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)
df['Burnout_Risk'] = (df['OverTime'] == 'Yes').astype(int) * df['DistanceFromHome'] / df['WorkLifeBalance']
df['Manager_Stability'] = df['YearsWithCurrManager'] / (df['YearsAtCompany'] + 1)
df['Engagement_Index'] = (df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction'] + df['JobInvolvement']) / 4
df['Career_Velocity'] = df['JobLevel'] / (df['TotalWorkingYears'] + 1)
df['Income_Growth_Gap'] = df['PercentSalaryHike'] - df.groupby('JobLevel')['PercentSalaryHike'].transform('median')
df['Loyalty_Index'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1)
travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
df['Travel_Burden'] = df['BusinessTravel'].map(travel_map)

df.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours', 'Avg_Role_Income'], inplace=True, errors='ignore')
print(f'Shape: {df.shape}')
print(f'Attrition Rate: {df["Attrition"].mean():.1%}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# OverTime vs Attrition
ot = df.groupby('OverTime')['Attrition'].mean() * 100
axes[0,0].bar(ot.index, ot.values, color=['#2ecc71', '#e74c3c'])
axes[0,0].set_title('Attrition Rate by OverTime')
axes[0,0].set_ylabel('Attrition %')

# Department
dept = df.groupby('Department')['Attrition'].mean() * 100
axes[0,1].barh(dept.index, dept.values, color='#3498db')
axes[0,1].set_title('Attrition by Department')

# Income distribution
for label, group in df.groupby('Attrition'):
    tag = 'Left' if label == 1 else 'Stayed'
    axes[0,2].hist(group['MonthlyIncome'], bins=30, alpha=0.6, label=tag)
axes[0,2].set_title('Income Distribution')
axes[0,2].legend()

# Compa Ratio vs Attrition
for label, group in df.groupby('Attrition'):
    tag = 'Left' if label == 1 else 'Stayed'
    axes[1,0].hist(group['Compa_Ratio'], bins=30, alpha=0.6, label=tag)
axes[1,0].set_title('Compa Ratio Distribution')
axes[1,0].legend()

# Burnout Risk
for label, group in df.groupby('Attrition'):
    tag = 'Left' if label == 1 else 'Stayed'
    axes[1,1].hist(group['Burnout_Risk'], bins=30, alpha=0.6, label=tag)
axes[1,1].set_title('Burnout Risk Distribution')
axes[1,1].legend()

# Promotion Stagnation
for label, group in df.groupby('Attrition'):
    tag = 'Left' if label == 1 else 'Stayed'
    axes[1,2].hist(group['Promotion_Stagnation'], bins=30, alpha=0.6, label=tag)
axes[1,2].set_title('Promotion Stagnation Distribution')
axes[1,2].legend()

fig.tight_layout()
plt.show()

In [ ]:
# Correlation with Attrition
numeric_df = df.select_dtypes(include=[np.number])
corr_with_attrition = numeric_df.corr()['Attrition'].drop('Attrition').sort_values()

fig, ax = plt.subplots(figsize=(8, 12))
corr_with_attrition.plot(kind='barh', ax=ax, color=corr_with_attrition.apply(lambda x: '#e74c3c' if x > 0 else '#2ecc71'))
ax.set_title('Feature Correlation with Attrition')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Model Training (XGBoost + Optuna)

In [ ]:
y = df['Attrition']
X = df.drop(columns=['Attrition'])
X = pd.get_dummies(X, columns=X.select_dtypes(include=['object']).columns, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Load pre-optimized params (from pipeline run) or use defaults
import json
from pathlib import Path

params_path = Path('../models/best_params.json')
if params_path.exists():
    with open(params_path) as f:
        best_params = json.load(f)
    print(f'Loaded optimized params: {best_params}')
else:
    best_params = {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.05}
    print('Using default params')

scale_w = (y_train == 0).sum() / (y_train == 1).sum()
model = xgb.XGBClassifier(**best_params, scale_pos_weight=scale_w, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

## 4. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'F2-Score: {fbeta_score(y_test, y_pred, beta=2):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stay', 'Leave'], yticklabels=['Stay', 'Leave'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global Summary
shap.summary_plot(shap_values, X_test, max_display=15)

In [ ]:
# Local: Highest Risk Employee
highest_risk_idx = np.argmax(y_proba)
print(f'Highest Risk: {y_proba[highest_risk_idx]:.1%}')
shap.force_plot(explainer.expected_value, shap_values[highest_risk_idx, :], X_test.iloc[highest_risk_idx, :])

## 6. Risk Scoring & Cost Model

In [ ]:
risk_df = pd.DataFrame({
    'Predicted_Probability': y_proba,
    'Actual': y_test.values,
    'MonthlyIncome': df.loc[X_test.index, 'MonthlyIncome'].values if 'MonthlyIncome' in df.columns else 0
}, index=X_test.index)

risk_df['Risk_Tier'] = pd.cut(risk_df['Predicted_Probability'], bins=[0, 0.3, 0.6, 1.0], labels=['Low', 'Medium', 'High'])
risk_df['Annual_Salary'] = risk_df['MonthlyIncome'] * 12
risk_df['Replacement_Cost'] = risk_df['Annual_Salary'] * 1.5
risk_df['Expected_Loss'] = risk_df['Predicted_Probability'] * risk_df['Replacement_Cost']

summary = risk_df.groupby('Risk_Tier', observed=True).agg(
    Count=('Predicted_Probability', 'count'),
    Avg_Risk=('Predicted_Probability', 'mean'),
    Total_Expected_Loss=('Expected_Loss', 'sum')
).round(2)

print(summary)
print(f'\nTotal Value at Risk: ${risk_df["Expected_Loss"].sum():,.0f}')

## 7. What-If Scenario Analysis

In [ ]:
def simulate_intervention(employee_idx, action, model=model, X_test=X_test):
    """Simulate a retention action and show risk change."""
    emp = X_test.iloc[[employee_idx]].copy()
    original_risk = model.predict_proba(emp)[0][1]
    
    if action == 'raise_10pct' and 'MonthlyIncome' in emp.columns:
        emp['MonthlyIncome'] *= 1.10
        emp['Compa_Ratio'] *= 1.10
    elif action == 'remove_overtime' and 'OverTime_Yes' in emp.columns:
        emp['OverTime_Yes'] = 0
        emp['Burnout_Risk'] = 0
    elif action == 'promote' and 'YearsSinceLastPromotion' in emp.columns:
        emp['YearsSinceLastPromotion'] = 0
        emp['Promotion_Stagnation'] = 0
    
    new_risk = model.predict_proba(emp)[0][1]
    print(f'Original Risk: {original_risk:.1%}')
    print(f'After "{action}": {new_risk:.1%}')
    print(f'Risk Reduction: {(original_risk - new_risk):.1%}')
    return original_risk, new_risk

# Example: simulate removing overtime for the highest risk employee
print('--- Scenario: Remove Overtime ---')
simulate_intervention(highest_risk_idx, 'remove_overtime')
print()
print('--- Scenario: 10% Raise ---')
simulate_intervention(highest_risk_idx, 'raise_10pct')
print()
print('--- Scenario: Promote ---')
simulate_intervention(highest_risk_idx, 'promote')